🇬🇧 [English](02_modeling.ipynb) · 🇪🇸 [Español](02_modeling_es.ipynb)

# Cohorte PBC: análisis de modelado

Esta notebook usa los módulos de producción mantenidos en `src/` para carga, split,
preprocesamiento, ajuste, selección de umbral, y métricas. Es ejecutable desde
`cirrhosis/` o desde `cirrhosis/notebooks/`.

El endpoint primario es **Estadio 3–4 versus Estadio 1–2**. Los modelos de cuatro
estadios exactos y ordinal acumulativo son secundarios. Esto es investigación
exploratoria sin validación externa ni temporal: **no para uso clínico, no es consejo
médico, y no reemplaza una biopsia ni la evaluación de un clínico**. Ver
[guía AASLD de PBC](https://www.aasld.org/practice-guidelines/primary-biliary-cholangitis)
y [guía EASL de PBC](https://easl.eu/publication/management-of-cholestatic-liver-diseases/).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents]
                    if (p / "src" / "data.py").exists() and (p / "data" / "raw" / "pbc.csv").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import events_per_variable, load_pbc_data, train_test_split_pipeline
from src.evaluation import (
    binary_metrics,
    bootstrap_metric_intervals,
    calibration_metrics,
    multiclass_metrics,
    select_operating_threshold,
)
from src.modeling import build_named_pipeline, fit_model


In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "pbc.csv"
frame = load_pbc_data(DATA_PATH)
MODELS = ("logistic", "random_forest", "hist_gradient_boosting")
print(f"Raíz del proyecto: {PROJECT_ROOT}; filas validadas={len(frame)}")
print(f"Modelos comparados: {MODELS}")


Raíz del proyecto: /home/datakrdo/Documents/portfolio/cirrhosis; filas validadas=418
Modelos comparados: ('logistic', 'random_forest', 'hist_gradient_boosting')


La notebook resuelve su raíz a partir de la presencia de `src/data.py` y el CSV
validado, así que la ubicación de ejecución no puede seleccionar silenciosamente un
dataset distinto.

La reproducibilidad importa porque las afirmaciones de un modelo sobre una cohorte
histórica pequeña son especialmente sensibles a la procedencia de los datos.

Solo se usan los loaders y estimadores de producción; la lógica central de
preprocesamiento o modelado nunca se copia dentro de celdas de la notebook.

In [3]:
split = train_test_split_pipeline(frame, test_size=0.2, random_state=41)
epv = events_per_variable(split.X_train, split.y_binary_train)
print({
    "train_rows": len(split.X_train), "test_rows": len(split.X_test),
    "excluded_predictor_columns": ["Stage", "ID", "N_Days", "Status", "Drug"],
    "added_predictor_columns": ["trial_cohort"],
    "train_test_index_overlap": len(set(split.X_train.index) & set(split.X_test.index)),
    "events_per_variable_train": round(epv, 3),
})
print("Conteos de endpoint en train:", split.y_binary_train.value_counts().sort_index().to_dict())
print("Conteos de endpoint en test:", split.y_binary_test.value_counts().sort_index().to_dict())
print("Conteos de cohorte en train:", split.X_train["trial_cohort"].value_counts().to_dict())
print("Conteos de cohorte en test:", split.X_test["trial_cohort"].value_counts().to_dict())


{'train_rows': 329, 'test_rows': 83, 'excluded_predictor_columns': ['Stage', 'ID', 'N_Days', 'Status', 'Drug'], 'added_predictor_columns': ['trial_cohort'], 'train_test_index_overlap': 0, 'events_per_variable_train': 5.625}
Conteos de endpoint en train: {0: 90, 1: 239}
Conteos de endpoint en test: {0: 23, 1: 60}
Conteos de cohorte en train: {'randomised': 249, 'registry': 80}
Conteos de cohorte en test: {'randomised': 63, 'registry': 20}


Los pacientes etiquetados se dividen una vez, estratificados sobre endpoint x
`trial_cohort` para que tanto la subcohorte randomised como la registry estén
representadas en train y test; `Drug` se excluye como columna de fuga (es un proxy casi
perfecto de la pertenencia a la cohorte, ver `01_eda.ipynb`) y `trial_cohort` se incluye
en su lugar. El events-per-variable en el fold de entrenamiento está por debajo de la
regla convencional de EPV >= 10, lo cual se reporta en lugar de ocultarse.

El tiempo de seguimiento y el status pueden codificar el outcome de la enfermedad
posterior a la evaluación basal e inflarían el rendimiento aparente; las etiquetas de
test deben permanecer intocadas hasta el reporte final.

Todos los imputadores/encoders/modelos se ajustan solo con filas de entrenamiento, el
umbral operativo se selecciona por cross-validation solo sobre el fold de entrenamiento
(sin split de validación manual), y el conjunto de test se evalúa una sola vez.

In [4]:
thresholds = {}
final_models = {}
test_probabilities = {}
for name in MODELS:
    pipeline = build_named_pipeline(name, split.X_train, random_state=41)
    threshold, tuned = select_operating_threshold(
        pipeline, split.X_train, split.y_binary_train.to_numpy(), random_state=41
    )
    thresholds[name] = threshold
    final_models[name] = tuned
    test_probabilities[name] = tuned.predict_proba(split.X_test)[:, 1]
print("Umbrales de decisión ajustados por CV (TunedThresholdClassifierCV, balanced_accuracy):")
print({k: round(v, 4) for k, v in thresholds.items()})


Umbrales de decisión ajustados por CV (TunedThresholdClassifierCV, balanced_accuracy):
{'logistic': 0.5472, 'random_forest': 0.538, 'hist_gradient_boosting': 0.5329}


Los tres modelos se entrenan a través de `build_named_pipeline` (preprocesamiento
ajustado por fold desde cero para cada modelo — mediana/one-hot para
`logistic`/`random_forest`, sin imputación y dtype categórico nativo para
`hist_gradient_boosting`) y su umbral operativo se elige con
`TunedThresholdClassifierCV`, que hace cross-validation solo sobre el fold de
entrenamiento — sin split de validación manual, ninguna etiqueta de test toca jamás la
selección de umbral.

Un baseline lineal transparente, un ensamble de árboles no lineal, y un booster de
gradiente con manejo nativo de ausencia de datos evalúan si relaciones más allá de
efectos aditivos — y más allá de lo que preserva la imputación por mediana — importan,
sin implicar causalidad ni diagnóstico individual.

Los tres modelos preespecificados pasan a la comparación held-out;
`hist_gradient_boosting` es el comparador primario (ver `README.md`), los otros son
baselines. Ningún modelo se selecciona en base al rendimiento en test.

In [5]:
held_out = {}
for name, probabilities in test_probabilities.items():
    metrics = binary_metrics(split.y_binary_test.to_numpy(), probabilities, threshold=thresholds[name])
    metrics["bootstrap_95_ci"] = bootstrap_metric_intervals(
        split.y_binary_test.to_numpy(), probabilities, threshold=thresholds[name],
        n_bootstrap=100, random_state=41
    )
    metrics["calibration"] = calibration_metrics(split.y_binary_test.to_numpy(), probabilities)
    held_out[name] = metrics
print(json.dumps(held_out, indent=2))
artifact = PROJECT_ROOT / "outputs" / "notebook_modeling_metrics.json"
artifact.write_text(json.dumps({"primary_endpoint": "Stage 3-4 vs Stage 1-2", "models": held_out}, indent=2) + "\n")
print(f"Se escribió {artifact}")


{
  "logistic": {
    "auroc": 0.777536231884058,
    "auprc": 0.9106872897288659,
    "sensitivity": 0.5833333333333334,
    "specificity": 0.7391304347826086,
    "threshold": 0.5472444899273095,
    "positive_rate": 0.4939759036144578,
    "brier_score": 0.19513379136813166,
    "bootstrap_95_ci": {
      "auroc": [
        0.6728385416666667,
        0.900969696969697
      ],
      "auprc": [
        0.8453864726812629,
        0.9611308762961732
      ],
      "sensitivity": [
        0.4576271186440678,
        0.7184029807130331
      ],
      "specificity": [
        0.575,
        0.9047727272727272
      ]
    },
    "calibration": {
      "brier_score": 0.19513379136813166,
      "fraction_of_positives": [
        0.5555555555555556,
        0.25,
        0.5,
        0.75,
        0.8888888888888888,
        0.625,
        0.625,
        1.0,
        1.0,
        1.0
      ],
      "mean_predicted_value": [
        0.2579688127044452,
        0.33730721688734194,
        0

AUROC, AUPRC, sensibilidad, especificidad, e intervalos bootstrap percentiles se
calculan sobre las predicciones de test intocadas; los intervalos pueden ser amplios
porque solo cerca de un quinto de las 412 filas etiquetadas se reservan como held-out.

La incertidumbre y el desbalance de clases hacen que un único punto estimado no sea
seguro para decisiones clínicas, especialmente en una cohorte de la era de tratamiento
sin validación externa.

Se reporta el conjunto completo de métricas held-out con intervalos, sin declarar
superioridad a partir de diferencias pequeñas, y nada de esto se usa ni se despliega
para la atención del paciente.

In [6]:
secondary = {}
multiclass_model = fit_model("multiclass_logistic", split.X_train, split.y_stage_train, random_state=41)
secondary["multiclass_logistic"] = multiclass_metrics(
    split.y_stage_test.to_numpy(), np.asarray(multiclass_model.predict(split.X_test))
)
ordinal_model = fit_model("ordinal_logistic", split.X_train, split.y_stage_train, random_state=41)
secondary["ordinal_logistic"] = multiclass_metrics(
    split.y_stage_test.to_numpy(), np.asarray(ordinal_model.predict(split.X_test))
)
print(json.dumps(secondary, indent=2))
artifact = PROJECT_ROOT / "outputs" / "notebook_modeling_metrics.json"
payload = json.loads(artifact.read_text())
payload["secondary"] = secondary
artifact.write_text(json.dumps(payload, indent=2) + "\n")


{
  "multiclass_logistic": {
    "macro_f1": 0.242966042966043,
    "balanced_accuracy": 0.2549401295557155,
    "quadratic_weighted_kappa": 0.34545454545454546
  },
  "ordinal_logistic": {
    "macro_f1": 0.257328278322926,
    "balanced_accuracy": 0.27106916181378005,
    "quadratic_weighted_kappa": 0.35185185185185186
  }
}


4801

Los estimadores de cuatro estadios exactos y ordinal acumulativo usan el mismo split
train/test y el mismo preprocesamiento de producción, y se resumen con macro F1,
balanced accuracy, y kappa cuadrática ponderada en lugar del umbral binario.

El ordenamiento de estadios tiene significado clínico, pero la ambigüedad entre
estadios adyacentes y las clases tempranas escasas limitan la certeza; el acuerdo
ordinal no es prueba de una estadificación válida.

Estos análisis siguen siendo secundarios y generadores de hipótesis; el endpoint
binario sigue siendo la única afirmación primaria.

In [7]:
preprocessor = final_models["logistic"].estimator_.named_steps["preprocess"]
feature_names = list(preprocessor.get_feature_names_out())
coefficients = final_models["logistic"].estimator_.named_steps["model"].coef_[0]
importance = pd.Series(coefficients, index=feature_names).sort_values(key=np.abs, ascending=False).head(12)
print("Coeficientes logísticos de mayor valor absoluto (la dirección es asociación del modelo, no causalidad):")
print(importance.round(3).to_string())
print("\nNota: hist_gradient_boosting (el comparador primario) no expone coef_ ni "
      "feature_importances_; sus explicaciones vienen de SHAP en 03_model_development.ipynb.")
limitations = [
    "418 filas fuente, 412 valores de Stage etiquetados, y ausencia de datos sustancial -- y estructural, no aleatoria.",
    "Cohorte histórica de Mayo 1974-1984; se desconoce el espectro y la transportabilidad entre eras de tratamiento.",
    "El events-per-variable en el fold de entrenamiento está por debajo de la regla EPV >= 10.",
    "Split único aleatorio sin validación temporal ni externa; las estimaciones de nested-CV y bootstrap-optimism "
    "en 03_model_development.ipynb son los números principales más defendibles.",
    "Las asociaciones pueden reflejar confusión, disponibilidad de mediciones, y pertenencia a la cohorte del ensayo.",
    "No es reemplazo de biopsia, diagnóstico, recomendación de tratamiento, ni despliegue clínico.",
]
print("\nLimitaciones:")
for item in limitations:
    print(f"- {item}")


Coeficientes logísticos de mayor valor absoluto (la dirección es asociación del modelo, no causalidad):
Hepatomegaly_Y    0.387
Platelets        -0.347
Hepatomegaly_N   -0.318
Copper            0.245
Spiders_Y         0.222
Edema             0.208
Age               0.172
Albumin          -0.155
Spiders_N        -0.153
Ascites_N         0.139
Tryglicerides     0.117
Sex_M            -0.113

Nota: hist_gradient_boosting (el comparador primario) no expone coef_ ni feature_importances_; sus explicaciones vienen de SHAP en 03_model_development.ipynb.

Limitaciones:
- 418 filas fuente, 412 valores de Stage etiquetados, y ausencia de datos sustancial -- y estructural, no aleatoria.
- Cohorte histórica de Mayo 1974-1984; se desconoce el espectro y la transportabilidad entre eras de tratamiento.
- El events-per-variable en el fold de entrenamiento está por debajo de la regla EPV >= 10.
- Split único aleatorio sin validación temporal ni externa; las estimaciones de nested-CV y bootstrap-optimism

Los rankings de coeficientes resumen un modelo ajustado después de un split, y son
sensibles a variables de laboratorio correlacionadas, la ausencia de datos, y el
tamaño de muestra; no son una afirmación estable de feature-importance.

La estadificación de PBC requiere contexto clínico y vías diagnósticas validadas; la
guía AASLD/EASL, el juicio del clínico, y las investigaciones apropiadas priman sobre
este modelo exploratorio.

Estos coeficientes se usan solo para generar hipótesis de investigación. Se requiere
validación externa y temporal más calibración antes de cualquier discusión
traslacional, y este análisis no es un reemplazo de biopsia ni es para uso clínico.